# NB02 - Exploration et Ingéstion des données par programmation

## Mise en place

Dans un premier temps vérifier que l'on utilise le bon **Catalogue** et **Schéma** pour éviter de devoir renseigner le chemin complet dans les requêtes.

`SELECT current_catalog();`

`SELECT current_database();`

et pour changer les catalogues et schémas par défaut utiliser la requête : 

```sql
USE CATALOG labuser_<username>;
USE SCHEMA sql_analytics;
```

In [0]:
%sql
SELECT current_catalog();


In [0]:
%sql
SELECT current_database();

In [0]:
%sql
USE CATALOG labuser_edadou;
USE SCHEMA sql_analytics;

## Exploration

Maintenant qu'on a séléctionné les bons catalogue et schéma, on peut explorer les différents éléments :
- volumes
- fichiers
- tables

### 1. Tables

On peut lister les tables du schéma `sql_analytics` avec la requête SQL suivante : `SHOW_TABLES`

In [0]:
%sql
SHOW TABLES;

Et faire n'importe quelle requête SQL sur les tables du schéma, par exemple on va requêter la table `test_ui` qu'on a créé précédement

In [0]:
%sql
select * from test_ui;

### 2. Volumes

On peut faire la même chose pour les volumes : 

`SHOW VOLUMES`

Et lister les fichiers d'un volume : 

`LIST '/Volumes/labuser_<username>/sql_analytics/source_files'`

In [0]:
%sql
SHOW VOLUMES;

In [0]:
%sql
LIST '/Volumes/labuser_edadou/sql_analytics/source_files';

## Ingestion

### 1. Création d'une table à partir d'un fichier CSV présent sur un volume

On va créer une table à partir du fichier de test qui a été importé.

On peut utiliser 2 méthodes :
- CTAS
- COPY INTO

#### CTAS — *Create Table As Select*

Le **CTAS** crée une table en une seule requête en se basant sur le résultat d'un `SELECT`. C'est la méthode la plus simple pour créer une table à partir d'un fichier : la table est créée et remplie en une seule étape.

```sql
CREATE OR REPLACE TABLE nom_table AS
SELECT ... FROM source;
```

- **Avantage** : rapide, en une seule instruction.
- **Limite** : le schéma est inféré à partir du `SELECT` (types de colonnes non contrôlés manuellement sauf si on définit le schéma explicitement).



In [0]:
%sql
LIST '/Volumes/labuser_edadou/sql_analytics/source_files';

Si le fichier de test est bien présent, et on veut un premier regard sur la structure du fichier.

In [0]:
%sql
SELECT * from text.`/Volumes/labuser_<username>/sql_analytics/source_files/test.csv`

C'est un fichier CSV avec une ligne d'en-tête et les valeurs sont séparées par un **;**

In [0]:
%sql
-- cette méthode ne permet de lire le fichier directement que si il n'y a pas d'en-tête et que les valeurs sont séparées par une virgule
SELECT * from csv.`/Volumes/labuser_<username>/sql_analytics/source_files/test.csv`

In [0]:
%sql
SELECT *
FROM read_files(
'/Volumes/labuser_<username>/sql_analytics/source_files/test.csv',
format => 'csv',
header => true,
delimiter => ';',
inferSchema => true
);

Maintenant que on peut lire le fichier corrêctement avec une requête **SELECT** on va utiliser la même requête pour créer la table : 

In [0]:
%sql
CREATE OR REPLACE TABLE test_prog AS
SELECT *
FROM read_files(
'/Volumes/labuser_<username>/sql_analytics/source_files/test.csv',
format => 'csv',
header => true,
delimiter => ';',
inferSchema => true
);

In [0]:
%sql
select * from test_prog;

La table a bien été créée sauf qu'il y a une colonne en plus que on ne veut pas **_rescued_data**

> **Qu'est-ce que `_rescued_data` ?**
> C'est une colonne automatiquement ajoutée par la fonction `read_files` (mode CSV) lorsque l'option `inferSchema => true` est activée. Elle contient les données qui n'ont pas pu être associées à une colonne existante du schéma détecté — par exemple des colonnes supplémentaires inattendues, des valeurs mal formatées ou des champs tronqués.
> Pour éviter cette colonne, on peut ajouter l'option `rescuedDataColumn => 'none'` dans `read_files`.

Donc on va modifier la requête pour définir manuellement le schéma : 


In [0]:
%sql
CREATE OR REPLACE TABLE test_prog AS
SELECT *
FROM read_files(
'/Volumes/labuser_<username>/sql_analytics/source_files/test.csv',
format => 'csv',
header => true,
delimiter => ';',
schema => 'id INT, name STRING, age INT'
);

select * from test_prog;

#### COPY INTO

Le **COPY INTO** copie les données d'un fichier vers une table Delta existante. Contrairement au CTAS, la table doit déjà exister et COPY INTO est **idempotent** : à chaque exécution, il ne charge que les fichiers qui n'ont pas encore été traités.

```sql
COPY INTO nom_table
FROM '/chemin/vers/les/fichiers'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'delimiter' = ';');
```

- **Avantage** : idempotent — relancer la commande ne duplique pas les données déjà chargées.
- **Avantage** : permet un chargement incrémental (nouveaux fichiers uniquement).
- **Limite** : la table doit être créée au préalable (`CREATE TABLE` avant le `COPY INTO`).


In [0]:
%sql
DROP TABLE IF EXISTS test_prog_copy_into;

CREATE TABLE test_prog_copy_into (
    id INT,
    nom STRING,
    age INT
);

COPY INTO test_prog_copy_into
FROM '/Volumes/labuser_<username>/sql_analytics/source_files/test.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'delimiter'=';', 'inferSchema'='true');

In [0]:
%sql
select * from test_prog_copy_into;